# Section F — Delta Lake Operations

Time travel, Change Data Feed, liquid clustering, and recovery drill for the FreshKart Lakehouse.

In [0]:
catalog = "fk_ce_2026_07_1378"

### F1 — Time Travel

**NOTE**: Run after Run 2 (Batch 2 product load).

In [0]:
# F1: Time travel on silver.products — run AFTER Run 2
# Get selling_price before Batch 2 and current selling_price for P004, P021, P050

# Check table history to find the version before Batch 2
print("=== silver.products history ===")
spark.sql(f"DESCRIBE HISTORY {catalog}.silver.products").select("version", "operation", "timestamp").show()

# Determine the version before Batch 2 (the second load)
# Batch 1 = initial load, Batch 2 = second load
# Update this after checking the history:
before_batch2_version = 0  # UPDATE after Run 2

spark.sql(f"""
CREATE TABLE {catalog}.ops.time_travel_check AS
SELECT
    p_curr.product_id,
    p_before.selling_price AS selling_price_before_batch2,
    p_curr.selling_price AS selling_price_current
FROM {catalog}.silver.products p_curr
LEFT JOIN {catalog}.silver.products VERSION AS OF {before_batch2_version} p_before
    ON p_curr.product_id = p_before.product_id
WHERE p_curr.product_id IN ('P004', 'P021', 'P050')
""")

display(spark.table(f"{catalog}.ops.time_travel_check"))

### F2 — Change Data Feed

**NOTE**: Run after Run 2 (Batch 2 product load).

In [0]:
# F2: Change Data Feed — run AFTER Run 2
# Create table with every CDF row from Batch 2 product load only

# Determine the CDF version range for Batch 2
# Update these after checking the history:
batch2_start = 1  # UPDATE after Run 2
batch2_end = 1    # UPDATE after Run 2

spark.sql(f"""
CREATE TABLE {catalog}.ops.product_changes_batch2 AS
SELECT
    product_id,
    change_type,
    mrp,
    selling_price,
    is_active,
    commit_version
FROM table_changes('{catalog}.silver.products', {batch2_start}, {batch2_end})
""")

display(spark.table(f"{catalog}.ops.product_changes_batch2"))

### F3 — Layout

In [0]:
# F3: Create gold.fact_order_line with liquid clustering on (order_date_key, store_id)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.gold.fact_order_line
USING DELTA
CLUSTER BY (order_date_key, store_id)
AS SELECT
    fo.order_date_key,
    fo.store_id,
    oi.order_id,
    oi.line_number,
    oi.product_id,
    oi.quantity,
    oi.unit_price,
    oi.discount_amount,
    oi.gross_amount,
    oi.net_amount,
    oi.updated_at
FROM {catalog}.silver.order_items oi
JOIN {catalog}.gold.fact_orders fo ON oi.order_id = fo.order_id
""")

# Optimize (re-run after Run 2 to satisfy "optimised at least once after Run 2")
spark.sql(f"OPTIMIZE {catalog}.gold.fact_order_line")

# Verify
detail = spark.sql(f"DESCRIBE DETAIL {catalog}.gold.fact_order_line")
print(f"Clustering columns: {detail.select('clusteringColumns').collect()[0][0]}")
print(f"Row count: {spark.table(f'{catalog}.gold.fact_order_line').count()}")
display(spark.table(f"{catalog}.gold.fact_order_line").limit(10))

### F4 — Recovery Drill

In [0]:
# F4: Recovery drill — deep clone, delete, restore via Delta operation
import time

# Step 1: Deep clone silver.orders to ops.orders_drill
spark.sql(f"DROP TABLE IF EXISTS {catalog}.ops.orders_drill")
spark.sql(f"CREATE TABLE {catalog}.ops.orders_drill DEEP CLONE {catalog}.silver.orders")

# Disable auto-compaction to avoid concurrent write conflicts during RESTORE
spark.sql(f"ALTER TABLE {catalog}.ops.orders_drill SET TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'false')")

# Record the version after clone (before delete)
clone_version = spark.sql(f"DESCRIBE HISTORY {catalog}.ops.orders_drill").select("version").collect()[0][0]
print(f"Clone version: {clone_version}")
print(f"Row count after clone: {spark.table(f'{catalog}.ops.orders_drill').count()}")

# Step 2: Delete S01 and S02 rows
spark.sql(f"DELETE FROM {catalog}.ops.orders_drill WHERE store_id IN ('S01', 'S02')")
print(f"Row count after delete: {spark.table(f'{catalog}.ops.orders_drill').count()}")

# Step 3: Restore to version before delete (Delta RESTORE, not re-insert)
# Retry to handle any lingering concurrent operations
for attempt in range(3):
    try:
        spark.sql(f"RESTORE TABLE {catalog}.ops.orders_drill TO VERSION AS OF {clone_version}")
        print(f"Restore succeeded on attempt {attempt + 1}")
        break
    except Exception as e:
        if attempt < 2:
            print(f"Restore attempt {attempt + 1} failed, retrying in 5s...")
            time.sleep(5)
        else:
            raise

print(f"Row count after restore: {spark.table(f'{catalog}.ops.orders_drill').count()}")

# Verify history shows clone, delete, and recovery
print("\n=== Delta history (should show CLONE, DELETE, RESTORE) ===")
display(spark.sql(f"DESCRIBE HISTORY {catalog}.ops.orders_drill").select(
    "version", "operation", "timestamp"
))